# TAT-C Propagation, OrbitPy Access Interface Implementation

Example which uses EOSE-API interfaces between TAT-C (orbit propagation) and OrbitPy (access calculations).

In [1]:
import warnings

import tempfile
import os, shutil
import csv

import geopandas as gpd
import pandas as pd

import json
from datetime import datetime, timedelta, timezone
from astropy.time import Time as AstroPy_Time
from shapely.geometry import box, mapping
from scipy.stats import hmean

from scipy.spatial.transform import Rotation as Scipy_Rotation

from joblib import Parallel, delayed

from tatc.schemas import Instrument as TATC_Instrument, Satellite as TATC_Satellite, TwoLineElements
from tatc.analysis import collect_orbit_track, OrbitCoordinate, OrbitOutput

from orbitpy.util import OrbitState as OrbitPy_OrbitState, Spacecraft as OrbitPy_Spacecraft, SpacecraftBus as OrbitPy_SpacecraftBus
from orbitpy.propagator import J2AnalyticalPropagator as OrbitPy_J2AnalyticalPropagator, SGP4Propagator as OrbitPy_SGP4Propagator
from orbitpy.coveragecalculator import GridCoverage as OrbitPy_GridCoverage, find_access_intervals as OrbitPy_find_access_intervals
from orbitpy.grid import Grid as OrbitPy_Grid

from eose.propagation import (
    PropagationSample,
    PropagationRecord,
    PropagationRequest,
    PropagationResponse,
)

from eose.orbits import GeneralPerturbationsOrbitState, Propagator
from eose.satellites import Satellite, SatelliteBus
from eose.utils import CartesianReferenceFrame, PlanetaryCoordinateReferenceSystem, Quaternion, FixedOrientation
from eose.geometry import Position

from eose.access import (
    AccessSample,
    AccessRecord,
    AccessRequest,
    AccessResponse,
)
from eose.grids import UniformAngularGrid

from instrupy import Instrument as InstruPy_Instrument

from eose.instruments import CircularGeometry, RectangularGeometry, BasicSensor

## Define mission parameters


In [4]:
iss_omm_str = '[{"OBJECT_NAME":"ISS (ZARYA)","OBJECT_ID":"1998-067A","EPOCH":"2024-06-07T09:53:34.728000","MEAN_MOTION":15.50975122,"ECCENTRICITY":0.0005669,"INCLINATION":51.6419,"RA_OF_ASC_NODE":3.7199,"ARG_OF_PERICENTER":284.672,"MEAN_ANOMALY":139.0837,"EPHEMERIS_TYPE":0,"CLASSIFICATION_TYPE":"U","NORAD_CAT_ID":25544,"ELEMENT_SET_NO":999,"REV_AT_EPOCH":45703,"BSTAR":0.00033759,"MEAN_MOTION_DOT":0.00019541,"MEAN_MOTION_DDOT":0}]'
iss_omm = json.loads(iss_omm_str)[0]

basic_sensor = BasicSensor(   id="Atom",
                        mass= 100.5,
                        volume= 0.75,
                        power= 150.0,
                        field_of_view = RectangularGeometry(angle_height=60.0, angle_width=30), #CircularGeometry(diameter=60.0)
                        orientation= list([0, 0.258819, 0, 0.9659258]), # +30 deg roll about x-axis (roll)
                        data_rate= 10.5,
                        bits_per_pixel= 16
                    )

satellites=[
        Satellite(
            id="ISS",
            orbit=GeneralPerturbationsOrbitState.from_omm(iss_omm),
            payloads=[
                basic_sensor
            ],
            satellite_bus=SatelliteBus(id="BlueCanyon XB16", orientation=FixedOrientation.NADIR_GEOCENTRIC)
        )
    ]

targets = UniformAngularGrid(
        delta_latitude=20, delta_longitude=20, region=mapping(box(-180, -50, 180, 50))
    ).as_targets()

mission_start = datetime(2024, 1, 1, tzinfo=timezone.utc)
mission_duration = timedelta(days=1)
propagate_time_step = timedelta(minutes=1)

## Run the TAT-C Propagator

In [5]:
def propagate_tatc(request: PropagationRequest) -> PropagationResponse:
    if request.propagator != Propagator.SGP4:
        raise RuntimeError("TAT-C only supports SGP4 propagator.")
    orbit_tracks = Parallel(-1)(
        delayed(collect_orbit_track)(
            TATC_Satellite(
                name=satellite.id,
                orbit=TwoLineElements(tle=satellite.orbit.to_tle()),
            ),
            TATC_Instrument(name="Instrument"),
            pd.date_range(
                request.start,
                request.start + request.duration,
                freq=request.time_step,
            ),
            coordinates=(
                OrbitCoordinate.ECI
                if request.frame == CartesianReferenceFrame.ICRF
                else OrbitCoordinate.ECEF
            ),
            orbit_output=OrbitOutput.POSITION_VELOCITY,
        )
        for satellite in request.satellites
    )
    return PropagationResponse(
        **request.model_dump(exclude="satellite_records"),
        satellite_records=[
            PropagationRecord(
                satellite_id=satellite.id,
                samples=orbit_tracks[i].apply(
                    lambda r: PropagationSample(
                        time=r.time,
                        frame=request.frame,
                        position=r.geometry.coords[0],
                        velocity=r.velocity.coords[0],
                    ),
                    axis=1,
                ),
            )
            for i, satellite in enumerate(request.satellites)
        ],
    )

propagate_request = PropagationRequest(
    satellites=satellites,
    start=mission_start,
    duration=mission_duration,
    frame=CartesianReferenceFrame.ICRF,
    propagator=Propagator.SGP4,
    time_step=propagate_time_step
)

#display(propagate_request.model_dump_json())

tatc_propagation_response = propagate_tatc(propagate_request)

#display(propagation_response.model_dump_json())

propagation_data = tatc_propagation_response.as_dataframe()

display(propagation_data)

,geometry,satellite_id,time,position,velocity
0,POINT Z (83.10851 48.53978 432820.67293),ISS,2024-01-01 00:00:00+00:00,"[-4499686.1202688, -232613.21550674032, 509165...","[-1677.6634157866658, -7252.659671818871, -180..."
1,POINT Z (88.15274 47.03603 432699.62950),ISS,2024-01-01 00:01:00+00:00,"[-4590028.087700935, -666915.9700864203, 49718...","[-1332.576798616102, -7218.509492576278, -2187..."
2,POINT Z (92.88653 45.30148 432541.96420),ISS,2024-01-01 00:02:00+00:00,"[-4659474.854577173, -1098182.7987153332, 4829...","[-981.4248032269486, -7151.498310442072, -2561..."
3,POINT Z (97.30894 43.36259 432355.86862),ISS,2024-01-01 00:03:00+00:00,"[-4707710.266126298, -1524450.42817976, 466472...","[-625.8040584505688, -7051.930817092407, -2922..."
4,POINT Z (101.43038 41.24458 432150.55101),ISS,2024-01-01 00:04:00+00:00,"[-4734514.636033979, -1943778.3089340637, 4478...","[-267.3323844160795, -6920.2597150227775, -327..."
...,...,...,...,...,...
1436,POINT Z (-146.62156 -50.42085 452107.98625),ISS,2024-01-01 23:56:00+00:00,"[2972775.053320527, -3178103.3234068537, -5248...","[4437.035162719276, 6104.25183656856, -1182.33..."
1437,POINT Z (-140.92203 -51.19414 452335.14649),ISS,2024-01-01 23:57:00+00:00,"[3232088.5905753896, -2804948.794978799, -5307...","[4203.456660244225, 6329.483090231624, -783.85..."
1438,POINT Z (-135.06176 -51.65632 452419.33520),ISS,2024-01-01 23:58:00+00:00,"[3476813.008662723, -2419133.0347507023, -5342...","[3950.9149924626354, 6526.134176716588, -381.8..."
1439,POINT Z (-129.12018 -51.79595 452358.25715),ISS,2024-01-01 23:59:00+00:00,"[3705843.9403711245, -2022397.7957213398, -535...","[3680.5378950486106, 6693.328594122815, 21.888..."


## Run OrbitPy Access Calculator

In this example the access calculations use the orbit-propagation results from TAT-C

In [ ]:
def access_orbitpy(request: AccessRequest) -> AccessResponse:

    # create a temporary directory to hold temporary files
    script_directory = os.path.dirname(os.path.abspath("__file__"))
    temp_dir = os.path.join(script_directory, "temp")
    os.makedirs(temp_dir, exist_ok=True)

    #### Enumerate and convert from EOSE-API satellites to OrbitPy satellite objects. ####
    # (Enumeration generates distinct orbit-instrument pairs for satellites equipped with multiple instruments.)
    OrbitPy_Satellites = []
    for satellite in request.satellites:
        for instru in satellite.payloads:
            if instru.id in request.payload_ids:
                instru_type = instru.type
                if instru_type == "BasicSensor":

                    if instru.field_of_view.type == "CircularGeometry":
                        instupy_fov_geom = {
                            "shape": "CIRCULAR",
                            "diameter": instru.field_of_view.diameter,
                        }
                    elif instru.field_of_view.type == "RectangularGeometry":
                        instupy_fov_geom = {
                            "shape": "RECTANGULAR",
                            "angleHeight": instru.field_of_view.angle_height,
                            "angleWidth": instru.field_of_view.angle_width,
                        }
                    else:
                        raise ValueError(
                            f"Only Circular and Rectangular geometries are supported and not {instru.field_of_view.type}"
                        )

                    # Convert orientation in Quaternion to Euler rotations
                    r = Scipy_Rotation.from_quat(list(instru.orientation))
                    (x, y, z) = r.as_euler(
                        "XYZ", degrees=True
                    )  # Conventions 'XYZ' are for intrinsic rotations (used by OrbitPy), while 'xyz' are for extrinsic rotations.

                    instrupy_sensor = InstruPy_Instrument.from_dict(
                        {
                            "@type": "Basic Sensor",
                            "orientation": {
                                "referenceFrame": "SC_BODY_FIXED",
                                "convention": "REF_FRAME_ALIGNED",
                            },
                            "fieldOfViewGeometry": instupy_fov_geom,
                            "orientation": {
                                "referenceFrame": "NADIR_POINTING",
                                "convention": "XYZ",
                                "xRotation": x,
                                "yRotation": y,
                                "zRotation": z,
                            },
                            "@id": instru.id,
                        }
                    )
                else:
                    raise ValueError(
                        f"{instru_type} instrument type is not supported. Only 'BasicSensor' instrument type is supported."
                    )

                tle = satellite.orbit.to_tle()

                orbit_state = OrbitPy_OrbitState.from_dict(
                    {
                        "tle": {
                            "tle_line0": "Unknown",
                            "tle_line1": tle[0],
                            "tle_line2": tle[1],
                        }
                    }
                )

                if (
                    hasattr(satellite, "satellite_bus") is False
                    or satellite.satellite_bus.orientation
                    == FixedOrientation.NADIR_GEOCENTRIC
                ):
                    orbitpy_sat_bus = OrbitPy_SpacecraftBus.from_dict(
                        {
                            "orientation": {
                                "referenceFrame": "Nadir_pointing",
                                "convention": "REF_FRAME_ALIGNED",
                            }
                        }
                    )
                else:
                    warnings.warn(
                        "OrbitPy only processes spacecraft-bus orientation aligned with the NADIR_GEOCENTRIC frame. To account for off-naidr instrument viewing, please specify the instrument orientation relative to the NADIR_GEOCENTRIC frame.",
                        UserWarning,
                    )

                sat = OrbitPy_Spacecraft(
                    _id=satellite.id,
                    orbitState=orbit_state,
                    spacecraftBus=orbitpy_sat_bus,
                    instrument=[instrupy_sensor],
                )

                OrbitPy_Satellites.append(sat)

    #### Format the Target points into OrbitPy Grid object ####
    lon = []
    lat = []
    target_id = []
    # iterate through the Target points
    for tp in request.targets:
        if tp.crs == PlanetaryCoordinateReferenceSystem.EPSG_4326 or tp.crs is None:
            lon.append(tp.position[0])
            lat.append(tp.position[1])
            target_id.append(tp.id)
        else:
            raise ValueError(
                f"{tp.crs} CRS is not supported by OrbitPy. Only 'EPSG_4326' CRS is supported."
            )

    row_to_target_id = (
        {}
    )  # Dictionary to map row numbers ('GP index' in OrbitPy) to target_id
    orbitpy_custom_grid = None
    with tempfile.NamedTemporaryFile(
        mode="w+t", delete=False, dir=temp_dir
    ) as grid_file:
        writer = csv.writer(grid_file)
        writer.writerow(["lat [deg]", "lon [deg]", "id"])

        for row_num, (lat_val, lon_val, target_id_val) in enumerate(
            zip(lat, lon, target_id)
        ):
            writer.writerow([lat_val, lon_val, target_id_val])
            row_to_target_id[row_num] = target_id_val

    orbitpy_custom_grid = OrbitPy_Grid.from_customgrid_dict(
        {"@type": "customGrid", "covGridFilePath": grid_file.name}
    )

    #### run propagation and coverage with OrbitPy ####
    step_size_s = request.time_step.total_seconds()
    if request.propagator != Propagator.J2:
        propagator = OrbitPy_J2AnalyticalPropagator.from_dict(
            {"@type": "J2 ANALYTICAL PROPAGATOR", "stepSize": step_size_s}
        )
    elif request.propagator != Propagator.SGP4:
        propagator = OrbitPy_SGP4Propagator.from_dict(
            {"@type": "SGP4 PROPAGATOR", "stepSize": step_size_s}
        )
    else:
        raise RuntimeError("OrbitPy only supports J2 and SGP4 propagators.")

    #### Convert request time to Julian Date UT1####
    utc_dt = request.start.astimezone(
        timezone.utc
    )  # Convert to UTC (if not already in UTC)
    astropy_utc_time = AstroPy_Time(
        utc_dt, scale="utc"
    )  # Convert to astropy Time object
    astropy_ut1_time = astropy_utc_time.ut1  # Convert to UT1 scale

    start_date_dict = {"@type": "JULIAN_DATE_UT1", "jd": astropy_ut1_time.jd}
    start_date = OrbitPy_OrbitState.date_from_dict(
        start_date_dict
    )  # assumed that the time scale is UT1.
    duration = request.duration.total_seconds() / 86400.0

    for orbitpy_sat in OrbitPy_Satellites:

        # run propagation with OrbitPy
        with tempfile.NamedTemporaryFile(
            mode="w+t", delete=False, dir=temp_dir
        ) as state_cart_file:  # store satellite states in a temporary file.
            propagator.execute(
                orbitpy_sat, start_date, state_cart_file.name, None, duration
            )

            # run access calculations with OrbitPy
            with tempfile.NamedTemporaryFile(
                mode="w+t", delete=False, dir=temp_dir
            ) as access_fl:
                cov_calc = OrbitPy_GridCoverage(
                    grid=orbitpy_custom_grid,
                    spacecraft=orbitpy_sat,
                    state_cart_file=state_cart_file.name,
                )
                instru_id = orbitpy_sat.get_instrument().get_id()
                cov_calc.execute(
                    instru_id=instru_id,
                    mode_id=None,
                    use_field_of_regard=True,
                    out_file_access=access_fl.name,
                    mid_access_only=False,
                )
                intervals_df = OrbitPy_find_access_intervals(access_fl.name)

                grouped_intervals = intervals_df.groupby("GP index")
                access_records = []  # record of accesses at each target point
                for gp_index, group in grouped_intervals:
                    # Iterate over each record in the group
                    access_sample = []
                    for index, row in group.iterrows():
                        access_start = request.start + timedelta(
                            seconds=row["Start time index"] * step_size_s
                        )
                        access_duration = timedelta(
                            seconds=row["Duration"] * step_size_s
                        )
                        # form access sample
                        access_sample.append(
                            AccessSample(
                                satellite_id=orbitpy_sat._id,
                                instrument_id=instru_id,
                                start=access_start,
                                duration=access_duration,
                            )
                        )
                    # Add access record
                    access_records.append(
                        AccessRecord(
                            target_id=row_to_target_id[gp_index], samples=access_sample
                        )
                    )

    # delete the temporary directory
    shutil.rmtree(temp_dir)

    return AccessResponse(
        **request.model_dump(exclude="target_records"), target_records=access_records
    )

'{"start":"2024-01-01T00:00:00Z","duration":"PT1H","satellites":[{"id":"ISS","orbit":{"object_name":"ISS (ZARYA)","object_id":"1998-067A","epoch":"2024-06-07T09:53:34.728000","mean_motion":15.50975122,"eccentricity":0.0005669,"inclination":51.6419,"ra_of_asc_node":3.7199,"arg_of_pericenter":284.672,"mean_anomaly":139.0837,"ephemeris_type":0,"classification_type":"U","norad_cat_id":25544,"element_set_no":999,"rev_at_epoch":45703,"bstar":0.00033759,"mean_motion_dot":0.00019541,"mean_motion_ddot":0.0},"payloads":[]}],"time_step":"PT1M","propagator":"sgp4","frame":"ICRF"}'

'{"start":"2024-01-01T00:00:00Z","duration":"PT1H","satellites":[{"id":"ISS","orbit":{"object_name":"ISS (ZARYA)","object_id":"1998-067A","epoch":"2024-06-07T09:53:34.728000","mean_motion":15.50975122,"eccentricity":0.0005669,"inclination":51.6419,"ra_of_asc_node":3.7199,"arg_of_pericenter":284.672,"mean_anomaly":139.0837,"ephemeris_type":0,"classification_type":"U","norad_cat_id":25544,"element_set_no":999,"rev_at_epoch":45703,"bstar":0.00033759,"mean_motion_dot":0.00019541,"mean_motion_ddot":0.0},"payloads":[]}],"time_step":"PT1M","propagator":"sgp4","frame":"ICRF","satellite_records":[{"satellite_id":"ISS","samples":[{"time":"2024-01-01T00:00:00Z","position":[-4499686.1202688,-232613.21550674032,5091657.311614182],"velocity":[-1677.6634157866658,-7252.659671818871,-1804.8077999287968]},{"time":"2024-01-01T00:01:00Z","position":[-4590028.087700935,-666915.9700864203,4971828.475405545],"velocity":[-1332.576798616102,-7218.509492576277,-2187.9548996394055]},{"time":"2024-01-01T00:02:00

,geometry,satellite_id,time,position,velocity
0,POINT Z (83.10851 48.53978 432820.67293),ISS,2024-01-01 00:00:00+00:00,"[-4499686.1202688, -232613.21550674032, 509165...","[-1677.6634157866658, -7252.659671818871, -180..."
1,POINT Z (88.15274 47.03603 432699.62950),ISS,2024-01-01 00:01:00+00:00,"[-4590028.087700935, -666915.9700864203, 49718...","[-1332.576798616102, -7218.509492576277, -2187..."
2,POINT Z (92.88653 45.30148 432541.96420),ISS,2024-01-01 00:02:00+00:00,"[-4659474.854577173, -1098182.798715333, 48293...","[-981.4248032269486, -7151.498310442072, -2561..."
3,POINT Z (97.30894 43.36259 432355.86862),ISS,2024-01-01 00:03:00+00:00,"[-4707710.266126301, -1524450.4281797595, 4664...","[-625.8040584505683, -7051.930817092405, -2922..."
4,POINT Z (101.43038 41.24458 432150.55101),ISS,2024-01-01 00:04:00+00:00,"[-4734514.636033979, -1943778.3089340634, 4478...","[-267.3323844160795, -6920.2597150227775, -327..."
...,...,...,...,...,...
56,POINT Z (-72.57352 -27.71768 439366.32380),ISS,2024-01-01 00:56:00+00:00,"[4529073.767911281, 3986627.0695678387, -31638...","[-1629.2524710910448, 5685.3328931624, 4849.06..."
57,POINT Z (-69.84962 -24.91917 438039.85643),ISS,2024-01-01 00:57:00+00:00,"[4421118.990028289, 4318448.275739423, -286593...","[-1967.880213649685, 5371.110358027979, 5077.8..."
58,POINT Z (-67.25735 -22.06222 436740.62396),ISS,2024-01-01 00:58:00+00:00,"[4293101.719886929, 4630672.72570865, -2554970...","[-2297.742415388213, 5032.3537458145065, 5283...."
59,POINT Z (-64.77708 -19.15611 435483.56620),ISS,2024-01-01 00:59:00+00:00,"[4145593.370171102, 4921873.335961587, -223237...","[-2617.3294734951587, 4670.580692719185, 5465...."


In [ ]:
request = AccessRequest(
    satellites=satellites,
    targets=targets,
    start=mission_start,
    duration=mission_duration,
    propagation_records=tatc_propagation_response.records,
    payload_ids=["Atom"]
)

display(request.model_dump_json())

access_response = access_orbitpy(request)

#display(access_response.model_dump_json())

access_data = access_response.as_dataframe()

#display(access_data)